# PyJST in Google Colab

This notebook installs PyJST from GitHub, runs its straight-channel and compression-corner demonstration cases, and displays pressure/convergence figures.

> **GPU note:** Colab's GPU runtime can be selected from **Runtime → Change runtime type**, but the current PyJST solver uses NumPy and therefore runs on the CPU. A GPU backend will be added separately.

In [ ]:
# Install the package and its optional plotting dependency from GitHub.
%pip install -q "git+https://github.com/VishalKandala/PyJST.git" "matplotlib>=3.7"

In [ ]:
from dataclasses import replace

from pyjst import (
    CartesianGrid,
    CompressionCornerCase,
    JSTParameters,
    freestream_initial_state,
    solve,
    uniform_supersonic_case,
)
from pyjst.postprocess import mach_number, plot_solution


## Straight channel: uniform Mach-2 flow

This exact-preservation case should have a pressure ratio of one everywhere and a zero residual.

In [ ]:
channel_case = replace(uniform_supersonic_case(), numerics=JSTParameters(max_iterations=1))
channel_grid = CartesianGrid(channel_case.grid)
channel_result = solve(freestream_initial_state(channel_grid, channel_case), channel_grid, channel_case)

print(f"normalized residual: {channel_result.residual_history[-1]:.3e}")
plot_solution(channel_result, channel_grid, channel_case, title="Straight channel: uniform Mach-2 flow");

## Mach-2, 10-degree compression corner

The body-fitted mesh captures the pressure rise and oblique shock downstream of the corner. The 500-iteration result is a visual demonstration rather than a fully converged validation study.

In [ ]:
corner_definition = CompressionCornerCase(nx=160, ny=80)
corner_case = replace(
    corner_definition.solver_case(),
    numerics=JSTParameters(cfl=0.4, cfl_initial=0.05, cfl_ramp_iterations=100, max_iterations=500),
)
corner_grid = corner_definition.grid()
corner_result = solve(freestream_initial_state(corner_grid, corner_case), corner_grid, corner_case)

print(f"normalized residual after {corner_result.iterations} iterations: {corner_result.residual_history[-1]:.3e}")
print(f"Mach-number range: {mach_number(corner_result.conservative, corner_grid, corner_case).min():.3f} to {mach_number(corner_result.conservative, corner_grid, corner_case).max():.3f}")
plot_solution(corner_result, corner_grid, corner_case, title="Compression corner: Mach 2, 10° deflection");